# Interactive Notebook

Connect to a TSD-based render server (VisRTX) and control it
interactively from Jupyter.  This notebook demonstrates:

- **Viewer**: streamed frame buffer with orbit / dolly / pan controls
- **Renderer & Camera**: set samples-per-pixel, denoise, camera pose, FOV
- **Frame export**: high-resolution JPEG/PNG snapshots with AOV support
- **Data tree panel**: browse and edit scene graph objects
- **Lights panel**: adjust light parameters in real time
- **Clip planes panel**: per-volume axis-aligned clipping
- **Transfer function panel**: colour ramp, opacity curve, presets
- **Load / save transfer functions**: JSON file support

In [ ]:
from IPython.display import display
from tsd_client import TSDSession

HOST = "127.0.0.1"
PORT = 49000

## 1: Connect

Opens a TCP connection to the TSD server and creates the session.
`TSDSession` bundles a `TSDClient`, an `OrbitTSDViewer`, and all the
interactive panels (data tree, lights, clip planes, transfer function).

Adjust `HOST` and `PORT` above to match your server.

In [ ]:
session = TSDSession(HOST, PORT)
client = session.client
print(f"Connected to {HOST}:{PORT}")

## 2: Viewer

Displays the live-streamed frame buffer from the server.

**Controls:**
- **Left-drag**: orbit (azimuth / elevation)
- **Right-drag** or **scroll**: dolly (zoom)
- **Middle-drag**: pan (shift look-at point)

In [ ]:
display(session.viewer)

## 3: Renderer parameters

Set ANARI renderer parameters via `client.set_renderer_param()`.
Common parameters include `pixelSamples` (SPP), `denoise`, and `ambientRadiance`.

Uses `GH_SET_RENDERER_PARAM` (message type 109) since GreenHorizon servers
manage renderers outside the TSD scene object pool.

In [ ]:
# Set samples per pixel
client.set_renderer_param("pixelSamples", 1)

# Other useful renderer params
client.set_renderer_param("denoise", False)
client.set_renderer_param("ambientRadiance", 0.0)

## 4: Frame export

Render the current view at a specified resolution and samples-per-pixel,
then save the result locally.  The viewer's frame config is automatically
restored after export.

Supported AOV channels: `color` (default), `depth`, `albedo`, `normal`,
`edges`, `objectId`, `primitiveId`, `instanceId`.

In [ ]:
client.export_frame("frame.png", width=800, height=600, spp=64)
client.export_frame("depth.png", aov="depth", depth_min=0, depth_max=20000.0)
client.export_frame("edges.png", aov="edges", invert=True)

## 5: Camera control

Set camera parameters directly: `position`, `direction`, `up`, `fovy`,
`apertureRadius`, etc.  Uses `SERVER_SET_OBJECT_PARAMETER` targeting the
camera in the TSD scene pool.

`set_camera_pose()` is a convenience wrapper that sets position, direction,
and up in a single call.

In [ ]:
# Set individual camera attributes
client.set_camera_attribute("fovy", 20.0)
client.set_camera_attribute("apertureRadius", 0.01)

# Or set position + direction + up in one call
session.client.set_camera_pose(
    position=(0.0, 1.4, 1.0),
    direction=(0.0, -0.5, -0.5),
    up=(0.0, 1.0, 0.0),
)

## 6: Data tree panel

Hierarchical view of the scene graph: layers, objects, parameters, and
metadata.  Click an object to see its parameters; editable values are sent
to the server via `SERVER_SET_OBJECT_PARAMETER`.

Object references (e.g. a volume's spatial field) are shown as expandable
child nodes.

In [ ]:
display(session.dt_panel)

### Scene graph manipulation

You can also manipulate scene objects programmatically via `SceneGraph`.
The example below rotates a transform node by 45° around Y.

In [ ]:
from time import sleep
sg = session.scene_graph
xform = sg.transform("default", "Material Orb")
xform.rotation = (0.0, 45.0, 0.0)
xform.commit(session.client)

## 7: Lights panel

Lists all lights in the scene (from objectDB or layer tree fallback) with
editable parameters: colour, intensity, position, direction, and falloff.
Changes are sent to the server immediately.

In [ ]:
display(session.light_panel)

## 8: Clip planes panel

Per-volume axis-aligned clip ranges.  Use the sliders to set the min/max
bounds along each axis.  Each volume's clip box is updated independently.

In [ ]:
display(session.clip_panel)

## 9: Transfer function panel

Interactive transfer function editor with:

- **Volume selector**: switch between volumes; the panel loads the existing
  TF from the server (colour ramp + opacity control points)
- **Colour ramp**: choose a preset colourmap or edit colour stops manually
- **Opacity curve**: drag control points to shape the opacity response
- **Auto-apply**: changes are sent to the server automatically (debounced)

The panel persists TF state both client-side (in-session cache) and
server-side (`_tfColorStops`, `_tfOpacityPoints` custom parameters) so it
survives volume switches and kernel restarts.

In [ ]:
display(session.tf_panel) 

## 10: Disconnect

Stops rendering, closes widgets, and drops the TCP session.  Uncomment the
cell below to disconnect; you can then re-run from **Section 1** to
reconnect.

In [ ]:
session.disconnect()
print("Disconnected.")